<a href="https://colab.research.google.com/github/atempvisit/learning-machine-learning/blob/main/Module_15_Logistic_Regression_using_Scikit_Learn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression

In [ ]:
df = pd.read_csv('titanic_data_updated.csv')
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
133,134,yes,second,"Weisz, Mrs. Leopold (Mathilde Francoise Pede)",female,29.0,1,0,228414,26.0000,NaN,S
363,364,no,third,"Asim, Mr. Adola",male,35.0,0,0,SOTON/O.Q. 3101310,7.0500,NaN,S
878,879,no,third,"Laleff, Mr. Kristo",male,NaN,0,0,349217,7.8958,NaN,S
335,336,no,third,"Denkoff, Mr. Mitto",male,NaN,0,0,349225,7.8958,NaN,S
233,234,yes,third,"Asplund, Miss. Lillian Gertrud",female,5.0,4,2,347077,31.3875,NaN,S


In [ ]:
df['Family_Size'] = df['SibSp']+df['Parch']+1
df['Cabin'] = df['Cabin'].fillna('Missing')

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
888,889,no,third,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,Missing,S,4,M
796,797,yes,first,"Leader, Dr. Alice (Farnham)",female,49.0,0,0,17465,25.9292,D17,S,1,D
557,558,no,first,"Robbins, Mr. Victor",male,NaN,0,0,PC 17757,227.5250,Missing,C,1,M
883,884,no,second,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.5000,Missing,S,1,M
579,580,yes,third,"Jussila, Mr. Eiriik",male,32.0,0,0,STON/O 2. 3101286,7.9250,Missing,S,1,M


In [ ]:
df['Deck'].value_counts()

,count
Deck,
M,687
C,59
B,47
D,33
E,32
A,15
F,13
G,4
T,1


In [ ]:
X = df.drop('Survived', axis=1)
y = df['Survived']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
692,693,third,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,Missing,S,1,M
481,482,second,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,Missing,S,1,M
527,528,first,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,1,C
855,856,third,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Missing,S,2,M
801,802,second,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,Missing,Q,1,M
258,259,first,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,Missing,C,1,M
736,737,third,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.0,1,3,W./C. 6608,34.3750,Missing,S,5,M
462,463,first,"Gee, Mr. Arthur H",male,47.0,0,0,111320,38.5000,E63,S,1,E


In [ ]:
y_train

,Survived
692,yes
481,no
527,no
855,yes
801,yes
...,...
359,yes
258,yes
736,no
462,no


In [ ]:
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()


X_train['Z_Score'] = (X_train['Age'] - mean_age)/std_age

musk = (abs(X_train['Z_Score']) <= 3)

X_train = X_train[musk]
y_train = y_train[musk]

In [ ]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck,Z_Score
855,856,third,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Missing,S,2,M,-0.815155
801,802,second,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M,0.082312
652,653,third,"Kalvik, Mr. Johannes Halvorsen",male,21.0,0,0,8475,8.4333,Missing,S,1,M,-0.608047
509,510,third,"Lang, Mr. Fang",male,26.0,0,0,1601,56.4958,Missing,S,1,M,-0.262867
18,19,third,"Vander Planke, Mrs. Julius (Emelia Maria Vande...",female,31.0,1,0,345763,18.0000,Missing,S,2,M,0.082312
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
544,545,first,"Douglas, Mr. Walter Donald",male,50.0,1,0,PC 17761,106.4250,C86,C,2,C,1.393995
476,477,second,"Renouf, Mr. Peter Henry",male,34.0,1,0,31027,21.0000,Missing,S,2,M,0.289420
258,259,first,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,Missing,C,1,M,0.358456
736,737,third,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.0,1,3,W./C. 6608,34.3750,Missing,S,5,M,1.255923


In [ ]:
X_train.shape

(573, 14)

In [ ]:
y_train.shape

(573,)

In [ ]:
X_train['Fare'].describe()

,Fare
count,573.000000
mean,33.909889
std,50.614307
min,0.000000
25%,8.050000
50%,15.045800
75%,33.000000
max,512.329200


In [ ]:
fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)
fare_IQR = fare_Q3 - fare_Q1

minimum = max(0, fare_Q1 - 1.5*fare_IQR)
maximum = fare_Q3 + 1.5*fare_IQR


X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

/tmp/ipykernel_1788/581778706.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)


In [ ]:
X_train['Fare'].describe()

,Fare
count,573.000000
mean,25.782271
std,22.044153
min,0.000000
25%,8.050000
50%,15.045800
75%,33.000000
max,70.425000


In [ ]:
#numerical values
p1 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('scaler', MinMaxScaler())
    ]
)

In [ ]:
#categorical values
p3 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoding', OneHotEncoder(sparse_output=False, drop='first',handle_unknown='ignore'))
    ]
)

# categories goes one after another by strictly following the order of the input
# categories=[['third', 'second', 'first'], ['male', 'female']]


categories=[['third', 'second', 'first']]

p4 = Pipeline(
    steps=[
         ('imputer', SimpleImputer(strategy='most_frequent')),
         ('encoder', OrdinalEncoder(categories=categories)),
        ('scaler', MinMaxScaler())
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline_1', p1, ['Age']),
        ('pipeline_2', p2, ['Fare', 'Family_Size']),
        ('pipeline_3', p3, ['Deck','Sex', 'Embarked']),
        ('pipeline_4', p4, ['Pclass'])
    ],
    remainder='drop'
)

preprocessor

ColumnTransformer(transformers=[('pipeline_1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline_2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Fare', 'Family_Size']),
                                ('pipeline_3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoding',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Deck', 'Sex', 'Embarked']),
                                ('pipeline_4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['third',
                                                                              'second',
                                                                              'first']])),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Pclass'])])

In [ ]:
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)

print(y_train)
print(y_test)

[1 1 0 1 0 1 0 0 0 1 0 0 0 1 1 0 1 0 0 1 1 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0
 0 0 1 0 1 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 1 0 1 0 1 1 1 1 0 0 0 0 0 0 0 1 0
 0 1 0 1 1 0 0 1 0 0 0 0 0 0 0 0 1 1 0 1 1 1 0 1 0 1 0 1 0 0 0 0 1 1 1 1 0
 0 0 0 0 0 1 1 0 1 1 0 0 0 1 0 1 1 0 0 1 0 1 0 0 1 1 1 0 0 0 0 0 0 0 1 1 0
 0 1 0 1 0 0 1 0 1 0 0 1 0 0 0 1 1 1 1 0 1 0 1 0 0 0 0 1 0 0 0 1 1 0 1 0 0
 0 0 0 0 1 1 0 0 0 0 1 1 0 0 0 0 1 1 0 0 1 1 0 0 0 1 1 1 0 0 1 1 1 1 0 0 0
 1 0 0 0 0 1 0 0 1 1 0 1 0 0 0 0 1 1 0 1 1 1 1 0 1 0 1 1 1 0 0 0 0 1 0 0 0
 0 1 1 0 0 1 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 1 1 1 0 0 0
 1 0 0 1 1 1 1 0 0 0 0 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 1 1 0 0 0 1 0 1 1 0 0
 0 0 0 1 0 0 1 0 1 0 0 1 1 0 1 0 1 0 1 0 0 0 0 1 0 1 0 0 0 1 0 1 1 0 1 1 0
 0 1 0 0 1 0 0 1 1 1 0 0 0 0 1 0 0 1 0 1 0 0 0 1 1 0 1 0 1 0 1 0 0 0 0 0 1
 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 0 1 1 0 0 0 1 1 1 1 1 1 1 1 0 1 0 1 0
 0 1 1 1 1 1 1 0 0 1 1 1 0 1 0 1 0 1 0 0 0 1 1 0 0 0 0 1 0 1 1 1 1 1 1 0 0
 0 1 0 1 0 1 1 0 0 1 0 0 

In [ ]:
lr_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000))
    ]
)

lr_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('pipeline_1',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age']),
                                                 ('pipeline_2',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Fare', 'Family_Size']),
                                                 ('pipeline_3',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(...
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Deck', 'Sex', 'Embarked']),
                                                 ('pipeline_4',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(categories=[['third',
                                                                                               'second',
                                                                                               'first']])),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Pclass'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [ ]:
lr_model.fit(X_train, y_train)

lr_model['model'].coef_ # weight value
lr_model['model'].intercept_ # biased value
lr_model['model'].classes_


array([0, 1])

In [ ]:
X_test

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
565,566,third,"Davies, Mr. Alfred J",male,24.0,2,0,A/4 48871,24.1500,Missing,S,3,M
160,161,third,"Cribb, Mr. John Hatfield",male,44.0,0,1,371362,16.1000,Missing,S,2,M
553,554,third,"Leeni, Mr. Fahim (""Philip Zenni"")",male,22.0,0,0,2620,7.2250,Missing,C,1,M
860,861,third,"Hansen, Mr. Claus Peter",male,41.0,2,0,350026,14.1083,Missing,S,3,M
241,242,third,"Murphy, Miss. Katherine ""Kate""",female,NaN,1,0,367230,15.5000,Missing,Q,2,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
880,881,second,"Shelley, Mrs. William (Imanita Parrish Hall)",female,25.0,0,1,230433,26.0000,Missing,S,2,M
91,92,third,"Andreasson, Mr. Paul Edvin",male,20.0,0,0,347466,7.8542,Missing,S,1,M
883,884,second,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.5000,Missing,S,1,M
473,474,second,"Jerwan, Mrs. Amin S (Marie Marthe Thuillard)",female,23.0,0,0,SC/AH Basle 541,13.7917,D,C,1,D


In [ ]:
y_pred = lr_model.predict(X_test)

y_pred

array([0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1,
       1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1,
       0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0,
       0, 1, 0])

In [ ]:
lr_model.predict_proba(X_test)

array([[0.85526967, 0.14473033],
       [0.92583961, 0.07416039],
       [0.76916699, 0.23083301],
       [0.92814804, 0.07185196],
       [0.37698946, 0.62301054],
       [0.44157914, 0.55842086],
       [0.25183676, 0.74816324],
       [0.59167268, 0.40832732],
       [0.48134655, 0.51865345],
       [0.78337807, 0.21662193],
       [0.8624713 , 0.1375287 ],
       [0.8168756 , 0.1831244 ],
       [0.33786725, 0.66213275],
       [0.70570024, 0.29429976],
       [0.14681691, 0.85318309],
       [0.83812407, 0.16187593],
       [0.46621184, 0.53378816],
       [0.86457955, 0.13542045],
       [0.79236436, 0.20763564],
       [0.23659906, 0.76340094],
       [0.86457955, 0.13542045],
       [0.1986873 , 0.8013127 ],
       [0.86629622, 0.13370378],
       [0.46090317, 0.53909683],
       [0.86160782, 0.13839218],
       [0.02667073, 0.97332927],
       [0.67630795, 0.32369205],
       [0.63487317, 0.36512683],
       [0.81566922, 0.18433078],
       [0.81175439, 0.18824561],
       [0.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')

Accuracy: 0.7653631284916201
Precision: 0.6753246753246753
Recall: 0.7536231884057971
